<a href="https://colab.research.google.com/github/SahilPurabiya/PythonForAstronomy/blob/main/MassOfGlobularCluster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install astroquery astropy scipy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
from astroquery.gaia import Gaia
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table
from scipy.constants import G

In [ ]:
ra_center = 301.5330 # degrees
dec_center = -46.2474  # degrees
distance_kpc = 8.830 # kpc
half_light_radius_pc = 1.9  # pc #For NGC 362 = 1.9

search_radius = 0.2  # degrees (cluster region)

In [ ]:
query = f"""
SELECT
    source_id,
    ra, dec,
    parallax,
    pmra, pmdec,
    radial_velocity
FROM gaiadr3.gaia_source
WHERE
    CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra_center}, {dec_center}, {search_radius})
    ) = 1
AND pmra IS NOT NULL
AND pmdec IS NOT NULL
"""

job = Gaia.launch_job(query)
results = job.get_results()

In [ ]:
pmra = results['pmra']
pmdec = results['pmdec']

pmra_mean = np.median(pmra)
pmdec_mean = np.median(pmdec)

# Select stars within 1 mas/yr radius in PM space
mask = np.sqrt((pmra - pmra_mean)**2 + (pmdec - pmdec_mean)**2) < 1.6

members = results[mask]

print("Number of probable members:", len(members))

In [ ]:
# Subtract systemic motion
pmra_rel = members['pmra'] - np.mean(members['pmra'])
pmdec_rel = members['pmdec'] - np.mean(members['pmdec'])

# Convert each component to km/s
v_ra = 4.74 * pmra_rel * distance_kpc
v_dec = 4.74 * pmdec_rel * distance_kpc

# Compute 1D dispersions
sigma_ra = np.std(v_ra)
sigma_dec = np.std(v_dec)

# 1D velocity dispersion (isotropic assumption)
sigma_1D = np.sqrt((sigma_ra**2 + sigma_dec**2)/2)

print("1D Velocity dispersion (km/s):", sigma_1D)

In [ ]:
# Convert half-light radius to meters
R_h_m = half_light_radius_pc * 3.086e16

# Convert dispersion to m/s
sigma_m = sigma_1D * 1000

# Virial mass
M_kg = (5 * R_h_m * sigma_m**2) / G

# Convert to solar masses
M_sun = M_kg / 1.989e30

#print("M =", M_sun)
mantissa, exponent = f"{M_sun:.2e}".split('e')
print(f"M = {float(mantissa):.2f} × 10^{int(exponent)} M☉")

In [ ]:
# Correct value (in solar masses)
M_correct = 2.52*10e5
signed_error = ((M_sun - M_correct) / M_correct) * 100
print(f"Signed Percentage Error = {signed_error:.4f} %")